In [ ]:
import csv
import io
import re
import time
import zipfile
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

### Bài toán bao hàm biến phân
tìm điểm $x$ thỏa mãn:

$$ 0 \in Ax + Bx $$



Với bài toán

$$
\min_{x\in C}\frac12\|Rx-b\|_2^2,
$$

chọn

$$
A=N_C,\qquad B(x)=R^\top(Rx-b).
$$

Khi đó thuật toán trở thành

$$
\boxed{
x^{k+1}
=P_C\bigl(x^k-\gamma R^\top(Rx^k-b)\bigr).
}
$$

Trong đó $P_C$ là phép chiếu mêtric lên tập $C$.
# THUẬT TOÁN FORWARD-BACKWARD SPLITTING (FBM)

Điều kiện FBM:

- $A$ là toán tử đa trị đơn điệu cực đại (tồn tại resolvent)
- $B$ là toán tử đơn trị thường được giả thiết $\beta$ cocoercive (0< $\beta$ < 1/2)
- $J_{\gamma A} = (I + \gamma A)^{-1}$ là toán tử giải của A
- $\gamma > 0$ là cỡ bước ($0<\gamma<2\beta$)

# Ý TƯỞNG THUẬT TOÁN

Với $\gamma>0$, ta có:

$$
\begin{aligned}
0\in Ax^\ast+Bx^\ast
&\iff -Bx^\ast\in Ax^\ast\\
&\iff -\gamma Bx^\ast\in\gamma Ax^\ast\\
&\iff x^\ast-\gamma Bx^\ast
\in x^\ast+\gamma Ax^\ast\\
&\iff x^\ast-\gamma Bx^\ast
\in (I+\gamma A)x^\ast\\
&\iff x^\ast\in (I+\gamma A)^{-1}
(x^\ast-\gamma Bx^\ast)\\
&\iff \boxed{x^\ast=J_{\gamma A}
(x^\ast-\gamma Bx^\ast)}.
\end{aligned}
$$

Trong đó:
$$
J_{\gamma A}=(I+\gamma A)^{-1}.
$$

trong đó $J_{\gamma A}=(I+\gamma A)^{-1}$.


Suy ra $x^\ast\in\operatorname{Fix}(T)$, với

$$
T=J_{\gamma A}\circ(I-\gamma B).
$$

Do đó, thuật toán lặp được xác định bởi:

$$
\boxed{
x^{k+1}=T(x^k)
=J_{\gamma A}(x^k-\gamma Bx^k),
\quad k\ge 0.
}
$$


Thuật toán FBM tách việc cập nhật tại mỗi bước lặp k thành hai bước nối tiếp:

## 1. Bước Forward:

   Từ điểm hiện tại $x_k$, thực hiện một bước cập nhật trực tiếp theo toán tử B:

   $$
   y_k = x_k - \gamma B(x_k)
   $$

## 2. Bước Backward:

   Từ điểm trung gian $y_k$, áp dụng toán tử toán tử giải của A để thu được điểm lặp tiếp theo:

   $$
   x_{k+1} = J_{\gamma A}(y_k) <=> x_{k+1}
   =
   J_{\gamma A}\left(x_k-\gamma B(x_k)\right)
   $$

Trong trường hợp A = $\partial g$, với g là một hàm lồi, resolvent của $A$ chính là toán tử proximal:

$$
J_{\gamma A}
=
J_{\gamma \partial g}
=
\operatorname{prox}_{\gamma g}
$$

Khi đó bước Backward có thể viết dưới dạng:

$$
x_{k+1}
=
\operatorname{prox}_{\gamma g}(y_k)
$$

Quá trình lặp được thực hiện cho đến khi sai khác giữa hai điểm liên tiếp đủ nhỏ, chẳng hạn:

$$
\|x_{k+1}-x_k\| < \mathrm{tol}
$$

với $\mathrm{tol} > 0$ là ngưỡng sai số cho trước.


### Mã giả Forward–Backward

**Bước 0 (Khởi tạo).**

Chọn $x^0\in\mathcal H$, $\gamma\in(0,2\beta)$,
$\varepsilon>0$, số vòng lặp tối đa $K_{\max}$.
Đặt $k=0$.

**Bước 1.** Tính điểm trung gian:

$$
y^k=x^k-\gamma B(x^k).
$$

**Bước 2.** Tính:

$$
x^{k+1}=J_{\gamma A}(y^k).
$$

**Bước 3.** Nếu

$$
\|x^{k+1}-x^k\|\leq\varepsilon
$$

hoặc $k+1=K_{\max}$, dừng thuật toán.

**Bước 4.** Đặt $k:=k+1$ và quay lại Bước 1.

In [ ]:
from FBM import fbm